In [0]:
from pyspark.sql.functions import *
from datetime import datetime

import ast

In [0]:
dbutils.widgets.text('notebook_params', '')

In [0]:
notebook_params = dbutils.widgets.get('notebook_params')

if not notebook_params:
    raise Exception('Missing notebook parameters')

notebook_params

"{'table_name': 'cartoes'}"

In [0]:
params = ast.literal_eval(notebook_params)

table_name = params.get('table_name')
businessdate_col = params.get('businessdate_col')

table_name, businessdate_col

('cartoes', None)

In [0]:
def read_source():
    reader = (
        spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.inferSchema', 'true')
        .option('cloudFiles.inferColumnTypes', 'true')
        .option('cloudFiles.format', 'csv')
        .option('header', 'true')
        .option('cloudFiles.schemaLocation', f'/Volumes/mobills/bronze/schemas/{table_name}_schema/')
        .option('cloudFiles.schemaEvolutionMode', 'addNewColumns')
    )

    return reader

def run_ingestion(df):
    return (
        df.writeStream
        .format('delta')
        .outputMode('append')
        .option('mergeSchema', 'true')
        .option('checkpointLocation', f'/Volumes/mobills/bronze/checkpoints/{table_name}_checkpoint/')
        .trigger(availableNow=True)
        .partitionBy(['_processdate', '_businessdate'])
        .toTable(f'mobills.bronze.{table_name}')
    )

In [0]:
reader = read_source()
df = reader.load(f'/Volumes/mobills/default/raw_data/{table_name}')

df = (
    df
    .withColumn('_processdate', lit(datetime.now().strftime('%Y%m%d')))
    .withColumn('_businessdate', date_format(col(f'{businessdate_col}'), 'yyyyMMdd') if businessdate_col else lit(datetime.now().strftime('%Y%m%d')))
    .withColumn('_ingesttime', current_timestamp())
    .withColumn('_sourcefile', col('_metadata.file_path'))
)

run_ingestion(df)